In [2]:
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import schwabdev as sd
import time as tm
import zarr
import shutil
import os

import warnings

warnings.filterwarnings('ignore', message="The data type .* does not have a Zarr V3 specification.*")
warnings.filterwarnings('ignore', message="Consolidated metadata is currently not part in the Zarr format 3 specification.*")

from main_classes.DataManager import DataManager as DM
from utility.lib_apiManagment import create_client as CC

dm = DM()
zarr_store = xr.open_dataset(dm.hot_db_path)
zarr_store

<xarray.Dataset> Size: 12GB
Dimensions:  (day: 29, time: 288, ident: 5094, qVar: 36, fVar: 18)
Coordinates:
  * day      (day) <U10 1kB '2025-09-14' '2025-09-15' ... '2025-10-18'
  * qVar     (qVar) <U29 4kB 'reference.htbRate' ... 'quote.postMarketPercent...
  * fVar     (fVar) <U28 2kB 'assetSubType' 'ssid' ... 'quote.closePrice'
  * time     (time) <U5 6kB '00:00' '00:05' '00:10' ... '23:45' '23:50' '23:55'
  * ident    (ident) <U5 102kB 'A' 'AA' 'AACB' 'AACI' ... 'ZYBT' 'ZYME' 'ZYXI'
Data variables:
    5m       (day, time, ident, qVar) float64 12GB ...
    1d       (day, ident, fVar) float64 21MB ...

In [3]:
for day in zarr_store['5m'].day.values:
    print(day,end=": ")
    day_marks = zarr_store['5m'].sel(day=day,qVar='quote.mark')
    day_marks_nan = np.sum(np.isnan(day_marks.values))
    day_marks_good = np.sum(~np.isnan(day_marks.values))
    print(f'10:00am Marks: {day_marks_good}/{day_marks_good + day_marks_nan} -- {day_marks_good/(day_marks_good + day_marks_nan)*100}%')

2025-09-14: 10:00am Marks: 71162/1467072 -- 4.850614012127558%
2025-09-15: 10:00am Marks: 261722/1467072 -- 17.83975155956899%
2025-09-16: 10:00am Marks: 1382141/1467072 -- 94.21084991057018%
2025-09-17: 10:00am Marks: 1399859/1467072 -- 95.41856159752214%
2025-09-18: 10:00am Marks: 767231/1467072 -- 52.296751625005456%
2025-09-19: 10:00am Marks: 523036/1467072 -- 35.65169262312961%
2025-09-20: 10:00am Marks: 1462314/1467072 -- 99.67568053919645%
2025-09-21: 10:00am Marks: 1404553/1467072 -- 95.73851862757928%
2025-09-22: 10:00am Marks: 1461888/1467072 -- 99.64664310954063%
2025-09-23: 10:00am Marks: 1461888/1467072 -- 99.64664310954063%
2025-09-24: 10:00am Marks: 1461858/1467072 -- 99.64459822012826%
2025-09-25: 10:00am Marks: 1458798/1467072 -- 99.43601950006543%
2025-09-26: 10:00am Marks: 1457783/1467072 -- 99.36683407494657%
2025-09-27: 10:00am Marks: 1456492/1467072 -- 99.27883566723378%
2025-09-28: 10:00am Marks: 1454992/1467072 -- 99.17659119661475%
2025-09-29: 10:00am Marks: 14

In [ ]:
DATE = '2025-09-29'
data_10 = zarr_store['5m'].sel(day=DATE,time='10:00').to_dataframe()
data_10 = data_10.pivot_table(index='qVar',columns='ident',values='5m')
marks = data_10.loc['quote.mark']
#list(marks.values)

In [ ]:
data = zarr_store['5m'].sel(day='2025-09-29',qVar='quote.mark')
df = data.to_dataframe().pivot_table(index='time',columns='ident',values='5m')